<a href="https://colab.research.google.com/github/uranovnass/task/blob/main/ml11_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import time
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

# Данные
data = load_breast_cancer()
X, y = data.data, data.target

# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Кросс-валидация
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

# Варианты
variants = {
    "Без PCA (30 признаков)": None,
    "PCA 80 %": 0.80,
    "PCA 90 %": 0.90,
    "PCA 95 %": 0.95
}

for name, variance in variants.items():

    if variance is None:
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=5000, random_state=42)
        )
        n_features = X.shape[1]

    else:
        model = make_pipeline(
            StandardScaler(),
            PCA(n_components=variance),
            LogisticRegression(max_iter=5000, random_state=42)
        )

        # Узнаем количество компонент после PCA
        model.fit(X_train, y_train)
        n_features = model.named_steps["pca"].n_components_

    # Время обучения
    start = time.perf_counter()
    model.fit(X_train, y_train)
    train_time = time.perf_counter() - start

    # Cross-validation
    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy"
    )

    # Test
    test_score = model.score(X_test, y_test)

    results.append({
        "Вариант": name,
        "Признаков": n_features,
        "Качество, CV": cv_scores.mean(),
        "Качество, test": test_score,
        "Время обучения": train_time
    })

results_df = pd.DataFrame(results)

print(results_df.round({
    "Качество, CV": 4,
    "Качество, test": 4,
    "Время обучения": 4
}))

                  Вариант  Признаков  Качество, CV  Качество, test  \
0  Без PCA (30 признаков)         30        0.9780          0.9825   
1                PCA 80 %          5        0.9692          0.9561   
2                PCA 90 %          7        0.9714          0.9474   
3                PCA 95 %         10        0.9824          0.9737   

   Время обучения  
0          0.0390  
1          0.0367  
2          0.0185  
3          0.0292  


## Уровень B

Для оценки влияния PCA использовалась модель Logistic Regression. Сравнивались четыре варианта: исходные 30 признаков, PCA с сохранением 80%, 90% и 95% дисперсии. Масштабирование и PCA выполнялись внутри Pipeline, что позволяет избежать утечки данных при кросс-валидации.

| Вариант | Признаков | Качество, CV | Качество, test | Время обучения |
|---|---:|---:|---:|---:|
| Без PCA (30 признаков) | 30 | 0.9780 | 0.9825 | 0.0390 с |
| PCA 80 % | 5 | 0.9692 | 0.9561 | 0.0367 с |
| PCA 90 % | 7 | 0.9714 | 0.9474 | 0.0185 с |
| PCA 95 % | 10 | 0.9824 | 0.9737 | 0.0292 с |

**Модель:** Logistic Regression.

**Пайплайн:** масштабирование → PCA → Logistic Regression.

Наилучшее качество на тестовой выборке показала модель без PCA — 0.9825. Однако PCA 95% позволил уменьшить число признаков с 30 до 10, сохранив высокое качество: 0.9737 на тестовой выборке. При этом время обучения уменьшилось с 0.0390 до 0.0292 секунды.

PCA 80% и PCA 90% сильнее сокращают количество признаков, но сопровождаются заметным снижением качества на тестовой выборке.

**Вывод:** PCA в данной задаче не даёт выигрыша по максимальному качеству, поскольку лучшая точность получена без PCA. Однако PCA 95% является хорошим компромиссом между количеством признаков, временем обучения и качеством модели. Поэтому PCA может быть оправдан, если важны уменьшение размерности и упрощение признакового пространства.